In [18]:
manual_q_override_raw = {
    "Cogent Economics and Finance": "Q2",
    "Cogent Business and Management": "Q2",
    "ISRA International Journal of Islamic Finance": "no-Q",
    "Journal of Asian Finance, Economics and Business": "no-Q",
    "Journal of Ecohumanism": "no-Q",
    "Journal of Population and Social Studies": "Q2",
    "Journal of Security and Sustainability Issues": "no-Q",
    "Review of International Geographical Education Online": "no-Q",
    "The Journal of Behavioral Science": "Q3",
    "Library Philosophy and Practice": "no-Q",
    "Systematic Reviews in Pharmacy": "no-Q",
    "Economics and Sociology": "Q2",
    "Economic Research-Ekonomska Istraživanja": "Q2",
    "Developmental Medicine and Child Neurology": "Q1",
    "Corporate Governance": "Q1",
    "Compensation and Benefits Review": "Q2",
    "Nonprofit Management and Leadership": "Q2",
    "Technology Analysis and Strategic Management": "Q2",
    "Leadership and Organization Development Journal": "Q1",
    "Social Sciences and Humanities Open": "Q1",
    "Journal of Accounting and Organizational Change": "Q1",
    "Journal of Nonprofit and Public Sector Marketing": "Q2",
    "Journal of Open Innovation: Technology, Market, and Complexity": "Q1",
    "International Journal of Financial Studies": "Q2",
    "International Journal of Productivity and Quality Management": "Q4",
    "International Journal of Trade and Global Markets": "Q4",
    "Journal of Sustainable Development of Energy, Water and Environment Systems": "Q2",
    "Corporate Social Responsibility and Environmental Management": "Q1",
    "Pertanika Journal of Social Sciences and Humanities": "Q2",
    "International Journal of Economics and Management": "Q3",
    "Journal of Management Development": "Q1",
    "Finance: Theory and Practice": "Q3",
    "Al-Ihkam: Jurnal Hukum dan Pranata Sosial": "Q1",
    "Vlakna a Textil": "Q4",
}

In [19]:
import re
import pandas as pd

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

def override_key(value):
    if pd.isna(value):
        return ""
    
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)

    # Samakan variasi umum
    text = text.replace("&", "and")
    text = text.replace("–", "-").replace("—", "-")

    # Hilangkan singkatan di akhir seperti (IJFS), (JPSS), [JPSS]
    text = re.sub(r"\s*\([a-z0-9\-]+\)\s*$", "", text)
    text = re.sub(r"\s*\[[a-z0-9\-]+\]\s*$", "", text)

    # Pertahankan hyphen dan colon
    text = re.sub(r"[^a-z0-9\s\-\:]", "", text)
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s*:\s*", ": ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [21]:
import pandas as pd
df_checkpoint2=pd.read_excel(r"D:\Proyek FEB\lengkapin data\sudah 40% pakai ini.xlsx")

In [22]:
manual_q_override = {
    override_key(journal): q
    for journal, q in manual_q_override_raw.items()
}

df_manual = df_checkpoint2.copy()  # atau ganti dengan nama dataframe checkpoint terakhirmu
df_manual["override_key"] = df_manual[COL_JOURNAL].apply(override_key)

override_count = 0

for idx, row in df_manual.iterrows():
    key = row["override_key"]
    
    if key in manual_q_override:
        new_q = manual_q_override[key]
        old_q = df_manual.at[idx, COL_Q]
        
        if old_q != new_q:
            df_manual.at[idx, COL_Q] = new_q
            override_count += 1
        
        if "q_source_checkpoint2" in df_manual.columns:
            df_manual.at[idx, "q_source_checkpoint2"] = "manual_verified_scimago_website"

print("Jumlah baris yang terkena manual override:", override_count)

Jumlah baris yang terkena manual override: 121


In [23]:
df_manual[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      572
Q2      558
?       324
no-Q    311
Q3      253
Q4      135
Name: count, dtype: int64

In [24]:
df_manual[df_manual["override_key"].isin(manual_q_override.keys())][
    [COL_JOURNAL, COL_Q]
].drop_duplicates().sort_values(COL_JOURNAL)

,Journal,SCOPUS Q
2015,Al-Ihkam: Jurnal Hukum dan Pranata Sosial,Q1
2019,COGENT ECONOMICS & FINANCE,Q2
163,Cogent Business & Management,Q2
611,Cogent Business and Management,Q2
288,Cogent Economics & Finance,Q2
287,Cogent Economics and Finance,Q2
1932,Compensation & Benefits Review,Q2
799,Corporate Governance,Q1
882,Corporate Governance (Bingley),Q1
809,Corporate Social Responsibility and Environmen...,Q1


In [25]:
df_final = df_manual.drop(columns=["override_key"])

output_file = "45%.xlsx"
df_final.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

Saved: 45%.xlsx
